___
# <center>Um catálogo de distribuições</center>
___

## Aula 10

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * reconhecer se um fenômeno é contagem ou medida, e escolher entre discreta e contínua;
 * dizer o que cada uma das seis distribuições da aula descreve;
 * desenhar qualquer uma delas em Python, mexendo nos parâmetros;
 * conferir contra o histograma da base se o modelo escolhido se sustenta.

Nada aqui pede fórmula decorada. O que este notebook treina é o olho: ver a
forma, associar ao fenômeno, e desconfiar quando o desenho não bate com os
dados.


___
<div id="indice"></div>

## Índice

- [Contagem ou medida](#tipo)

- [As discretas](#discretas)

- [As contínuas](#continuas)

- [O modelo bate com a base?](#conferir)

- [Onde a suposição quebra](#quebra)

- [RESUMO](#resumo)


___
<div id="tipo"></div>

# Contagem ou medida

A primeira decisão é sempre a mesma, e não precisa de conta nenhuma:

| pergunta | tipo | gráfico |
|---|---|---|
| quantos? | **discreta** | barras separadas |
| quanto? | **contínua** | curva |

Cada barra de uma discreta é uma probabilidade de verdade, e as barras somam 1.
Numa contínua a altura não é probabilidade: **a probabilidade é a área**, e por
isso a pergunta é sempre por faixa.


In [ ]:
import numpy as np
import pandas as pd
from plotnine import *
from scipy import stats

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)


[Volta ao Índice](#indice)


___
<div id="discretas"></div>

# As discretas

Três, na ordem em que aparecem na aula. Repare que o código é o mesmo nas três,
mudando só a distribuição do `scipy`.


**Bernoulli.** Um sim ou não. Um recurso: provido ou não provido.


In [ ]:
p = 0.30

bernoulli = pd.DataFrame({"x": [0, 1], "prob": [1 - p, p]})

(
    ggplot(bernoulli, aes(x="factor(x)", y="prob"))
    + geom_col(fill="#12996f", width=0.5)
    + labs(x="provido (1) ou não (0)", y="probabilidade",
           title="Bernoulli com p = 0,30")
)


**Binomial.** Quantos sucessos em `n` tentativas. O escritório interpõe 10
recursos: quantos serão providos?


In [ ]:
n, p = 10, 0.30

binomial = pd.DataFrame({"x": range(n + 1)})
binomial["prob"] = stats.binom.pmf(binomial["x"], n, p)

(
    ggplot(binomial, aes(x="x", y="prob"))
    + geom_col(fill="#12996f")
    + scale_x_continuous(breaks=range(n + 1))
    + labs(x="recursos providos", y="probabilidade",
           title="Binomial com n = 10 e p = 0,30")
)


**Poisson.** Quantas ocorrências num período, sem teto natural. Quantas ações
novas chegam na vara hoje?


In [ ]:
lam = 4

poisson = pd.DataFrame({"x": range(15)})
poisson["prob"] = stats.poisson.pmf(poisson["x"], lam)

(
    ggplot(poisson, aes(x="x", y="prob"))
    + geom_col(fill="#12996f")
    + labs(x="ações novas no dia", y="probabilidade",
           title="Poisson com λ = 4")
)


**✍️ Agora você.** Mexa nos parâmetros e olhe o que muda de FORMA, não de números. Rode a binomial com p = 0,05 e depois com p = 0,50, e a Poisson com λ = 1 e depois com λ = 10. Onde fica o pico em cada caso?


In [ ]:
for parametro in [0.05, 0.50]:
    d = pd.DataFrame({"x": range(11)})
    d["prob"] = stats.binom.pmf(d["x"], 10, parametro)
    print(f"binomial p={parametro}: pico em {d.loc[d['prob'].idxmax(), 'x']}, "
          f"média {10 * parametro}")

for parametro in [1, 10]:
    d = pd.DataFrame({"x": range(25)})
    d["prob"] = stats.poisson.pmf(d["x"], parametro)
    print(f"poisson λ={parametro}: pico em {d.loc[d['prob'].idxmax(), 'x']}, "
          f"média {parametro}")


O pico da binomial fica em torno de $n \times p$, e o da Poisson em torno de
$\lambda$. Nas duas, a média é onde a massa se concentra, e é só isso que
precisa ficar.


[Volta ao Índice](#indice)


___
<div id="continuas"></div>

# As contínuas

Aqui o gráfico é uma curva, e a altura dela **não** é probabilidade. Repare no
eixo y: em algumas curvas ele passa de 1, o que seria impossível se fosse
probabilidade.


In [ ]:
grade = pd.DataFrame({"x": np.linspace(0, 120, 400)})

curvas = pd.concat([
    grade.assign(dens=stats.uniform.pdf(grade["x"], 20, 60), qual="uniforme"),
    grade.assign(dens=stats.expon.pdf(grade["x"], scale=20), qual="exponencial"),
    grade.assign(dens=stats.norm.pdf(grade["x"], 60, 12), qual="normal"),
])

(
    ggplot(curvas, aes(x="x", y="dens"))
    + geom_area(fill="#bfe6d5")
    + geom_line(size=0.8)
    + facet_wrap("qual", scales="free_y")
    + labs(x="dias", y="densidade", title="As três contínuas da aula")
)


- **uniforme**: um retângulo. Nenhum valor da faixa é mais provável que outro.
- **exponencial**: começa alta e cai. Muitos casos rápidos, uma cauda longa.
- **normal**: simétrica, com pico no meio.

A probabilidade de uma faixa é a **área** debaixo da curva, e o `scipy` já
entrega isso pronto com o `.cdf`.


In [ ]:
# P(o laudo sair entre 50 e 70 dias), na normal de média 60 e desvio 12
stats.norm.cdf(70, 60, 12) - stats.norm.cdf(50, 60, 12)


**✍️ Agora você.** Na mesma normal, calcule a probabilidade de o laudo sair em mais de 90 dias, e a de sair em exatamente 60 dias.


In [ ]:
print("mais de 90 dias:", 1 - stats.norm.cdf(90, 60, 12))
print("exatamente 60 dias:", stats.norm.cdf(60, 60, 12) - stats.norm.cdf(60, 60, 12))


Zero. Não é um defeito da conta: em variável contínua, a probabilidade de
**qualquer** ponto isolado é zero mesmo. Por isso, aqui, `>` e `>=` dão o mesmo
número, e em discreta não dão.


[Volta ao Índice](#indice)


___
<div id="conferir"></div>

# O modelo bate com a base?

Escolher uma distribuição é **supor** algo sobre o fenômeno, e suposição se
confere. O juiz é o histograma.


In [ ]:
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

URL = "https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados"

criminal = pd.read_csv(f"{URL}/tjsp_cjsg_criminal.csv")

# Fora as linhas sem regime: não dá para calcular proporção de regime
# em acórdão que não informou regime nenhum.
penas = criminal.dropna(subset=["regime_inicial"])

penas.shape


In [ ]:
penas["pena_anos"].describe()


Média e mediana bem diferentes já avisam: a distribuição não é simétrica. Vamos
desenhar o histograma com a normal por cima, usando a média e o desvio da
própria base.


In [ ]:
media = penas["pena_anos"].mean()
desvio = penas["pena_anos"].std()

grade = pd.DataFrame({"x": np.linspace(0, penas["pena_anos"].max(), 300)})
grade["dens"] = stats.norm.pdf(grade["x"], media, desvio)

(
    ggplot()
    + geom_histogram(penas, aes(x="pena_anos", y="..density.."),
                     bins=30, fill="#dcdcdc", color="white")
    + geom_line(grade, aes(x="x", y="dens"), color="#e50505", size=1)
    + labs(x="pena (anos)", y="densidade",
           title="A pena segue uma normal?")
)


**Não segue.** A normal é simétrica e desce para os dois lados; a pena tem um
piso em zero, uma concentração nos valores baixos e uma cauda comprida à
direita. A curva vermelha chega a prever pena negativa, que não existe.

É o mesmo desenho da **exponencial** da seção anterior, e é assim que quase todo
valor jurídico se comporta: pena, tempo de tramitação, indenização, honorários.


**✍️ Agora você.** Uma regra rápida para desconfiar: numa normal, cerca de 68% dos casos ficam a um desvio padrão da média. Calcule essa proporção na base e compare com os 68% que o modelo promete.


In [ ]:
dentro = penas["pena_anos"].between(media - desvio, media + desvio).mean()

print(f"o modelo promete: 68%")
print(f"a base entrega:   {dentro:.1%}")


A diferença é grande, e é uma **falsificação**: o modelo prometeu um número, os
dados entregaram outro. É assim que se descarta uma distribuição, e não por
opinião sobre a forma do gráfico.


[Volta ao Índice](#indice)


___
<div id="quebra"></div>

# Onde a suposição quebra

A binomial exige três coisas: número fixo de tentativas, mesma probabilidade em
todas, e **independência**. A terceira é a que mais cai por terra em problema
jurídico.


**✍️ Agora você.** Escreva, numa frase de comentário, um caso jurídico em que contar sucessos em n tentativas NÃO é binomial porque as tentativas se influenciam.


In [ ]:
# Exemplo: dez recursos do mesmo escritório, sobre a mesma tese, julgados pela
# mesma câmara. Se o primeiro é provido, os outros passam a ter mais chance,
# porque o que decide não é o acaso e sim o entendimento da câmara sobre a tese.
# Usar binomial aqui subestima muito a chance dos extremos, dez providos ou
# nenhum, que são justamente os cenários que interessam a quem recorre.


O mesmo raciocínio vale para as outras: a Poisson supõe que as ocorrências não
se aglomeram, e um mutirão de ações de um mesmo escritório quebra isso na hora.


[Volta ao Índice](#indice)


___
<div id="resumo"></div>

# RESUMO

1. **Contagem** vira barra e é discreta; **medida** vira curva e é contínua.

2. Discretas: Bernoulli (um sim ou não), Binomial (quantos de `n`) e Poisson
   (quantos no período, sem teto).

3. Contínuas: uniforme (sem preferência), exponencial (tempo de espera, cauda
   longa) e normal (soma de muitas causas).

4. Em contínua a probabilidade é **área**, e a de um ponto é zero.

5. Escolher a distribuição é **supor**. O histograma da base é quem decide se a
   suposição fica de pé, e dinheiro e tempo quase nunca são normais.


[Volta ao Índice](#indice)
